# 01b — Build Vector Search Index over LEDGAR Provisions

Consumes the Delta table from `01_nr_load_ledgar_to_delta` and produces a Databricks
Vector Search index for the agent's semantic retrieval (RAG) tool in `02_agent`.

1. Build retrieval corpus from the **train split only** (keeps test split out — no leakage during eval)
2. Enable Change Data Feed (required for Delta Sync indexes)
3. Create Vector Search endpoint + Delta Sync index with managed embeddings
4. Validate with sample similarity searches

In [0]:
# Configuration Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.dropdown(
    name="corpus_mode",
    defaultValue="Sampled (~8k rows)",
    choices=["Sampled (~8k rows)", "Full train split (~60k rows)"],
    label="Retrieval Corpus Size"
)
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en", "Embedding Model Endpoint")

In [0]:
# Install Vector Search Library
%pip install databricks-vectorsearch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Read Configurations & Set Variables
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
CORPUS_MODE = dbutils.widgets.get("corpus_mode")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint")

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.ledgar_vs_source"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.ledgar_provisions_index"   # must be a 3-part UC name

print(f"Corpus table: {SOURCE_TABLE}")
print(f"Index:        {INDEX_NAME}")

Corpus table: workspace.default.ledgar_vs_source
Index:        workspace.default.ledgar_provisions_index


In [0]:
# Build Retrieval Corpus (Train Split)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("default.ledgar_lexglue")

corpus_df = (
    df.filter(F.col("split") == "train")
      # category_label is an array<string> (cell 4 of 01 does x[0] on 'gold')
      .withColumn("category", F.col("category_label").getItem(0))
      .filter(F.trim(F.col("provision_text")) != "")
      # provisions are short paragraphs; no chunking needed. Truncate defensively.
      .withColumn("provision_text", F.substring("provision_text", 1, 8000))
      .select("provision_id", "provision_text", "category", "text_length")
)

if CORPUS_MODE.startswith("Sampled"):
    print("Sampled mode: capping at 80 rows per category (~8k rows, all 100 labels kept)")
    w = Window.partitionBy("category").orderBy(F.rand(seed=42))
    corpus_df = (
        corpus_df
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") <= 80)
        .drop("rn")
    )
else:
    print("Full mode: indexing entire train split")

print(f"Corpus rows: {corpus_df.count()}")
display(corpus_df.limit(10))

Sampled mode: capping at 80 rows per category (~8k rows, all 100 labels kept)
Corpus rows: 7861


provision_id,provision_text,category,text_length
51145,"This Agreement shall be binding upon each of the Secured Parties, each of the Credit Parties hereto and their respective successors and assigns.",Assigns,144
32450,"This Agreement shall be binding upon, and inure to the benefit of the parties hereto, their respective legal representatives, successors, and assigns.",Assigns,150
31281,"This Plan shall be binding upon the heirs, executors, administrators, successors and assigns of the parties, including each Participant, present and future (except that no successor to the Employer shall be considered a Plan sponsor unless that successor adopts this Plan).",Assigns,273
51372,"This Sublease (including the exhibits hereto which are hereby made a part hereof) contains the entire agreement between the parties, and any agreement hereafter made shall be ineffective to change, modify or discharge it in whole or in part unless such agreement is in writing and signed by the party against whom enforcement of the change, modification or discharge is sought. This Sublease shall bind and inure to the benefit of the parties hereto and their respective successors and their respective assigns.",Assigns,513
44738,"This Agreement shall inure to the benefit of, and shall be binding upon and enforceable against, the Loan Parties, the Credit Parties, the Swap Counterparties and Collateral Agent and their respective successors and permitted assigns.",Assigns,234
13407,"This Agreement shall be binding upon and inure to the benefit of the First Lien Administrative Agent, the other First Lien Claimholders, the Second Lien Collateral Trustee, the other Second Lien Claimholders, the Company and the other Grantors, and their respective successors and assigns from time to time. If either of the First Lien Administrative Agent or the Second Lien Collateral Trustee resigns or is replaced pursuant to the First Lien Loan Documents or the Second Lien Documents, as applicable, its successor and/or assign shall be deemed to be a party to this Agreement and shall have all the rights of, and be subject to all the obligations of, this Agreement. No provision of this Agreement will inure to the benefit of a bankruptcy trustee, debtor-in-possession, creditor trust or other representative of an estate or creditor of any Grantor, including where any such bankruptcy trustee, debtor-in-possession, creditor trust or other representative of an estate is the beneficiary of a Lien securing Collateral by virtue of the avoidance of such Lien in an Insolvency or Liquidation Proceeding.",Assigns,1108
52095,"This Agreement shall be binding upon the First Lien Agents, the Senior Lenders, the Second Priority Agents, the Second Priority Secured Parties and their respective permitted successors and assigns.",Assigns,198
49733,"All of the terms and provisions of this Third Amendment shall be binding upon and inure to the benefit of the parties hereto, their respective successors, assigns and legal representatives.",Assigns,189
9540,"This Agreement shall be binding upon the Senior Representatives, the Senior Secured Parties, the Second Priority Representative, the Second Priority Debt Parties, the Borrowers, the other Grantors party hereto and their respective successors and permitted assigns.",Assigns,264
1856,"This Agreement shall be binding upon the First Lien Representatives, the First Lien Secured Parties, the other First Lien Secured Parties, the Second Lien Representative, the Second Lien Secured Parties, the other Second Lien Secured Parties, the Third Lien Representative, the Third Lien Secured Parties, the other Third Lien Secured Parties, the Company and the other Grantors, and their respective successors and assigns. If any of the First Lien Representatives, the First Lien Collateral Agents, the Second Lien Representative or the Second Lien Collateral Agent, the Third Lien Representative or the Third Lien Collateral Agent resigns or is replaced pursuant to the First Lien D

In [0]:
# Write Corpus Table, Enable Change Data Feed
(
    corpus_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SOURCE_TABLE)
)

spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Created {SOURCE_TABLE} with Change Data Feed enabled")

Created workspace.default.ledgar_vs_source with Change Data Feed enabled


In [0]:
# Create Vector Search Endpoint
import time
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

def endpoint_exists(vsc, name):
    try:
        vsc.get_endpoint(name)
        return True
    except Exception:
        return False

if not endpoint_exists(vsc, VS_ENDPOINT):
    print(f"Creating endpoint {VS_ENDPOINT}...")
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")

for _ in range(60):
    status = vsc.get_endpoint(VS_ENDPOINT).get("endpoint_status", {}).get("state", "")
    print(f"Endpoint state: {status}")
    if status == "ONLINE":
        break
    time.sleep(30)
else:
    raise TimeoutError(f"Endpoint {VS_ENDPOINT} did not come online")

/home/spark-06f4ead2-a9a0-4281-81f3-7e/.ipykernel/72864/command-8112000843654316-3120051421:3: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Endpoint state: ONLINE


In [0]:
# Create Delta Sync Index 
def index_exists(vsc, endpoint, name):
    try:
        vsc.get_index(endpoint_name=endpoint, index_name=name)
        return True
    except Exception:
        return False

if not index_exists(vsc, VS_ENDPOINT, INDEX_NAME):
    print(f"Creating index {INDEX_NAME}...")
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",   # manual sync; cheaper than CONTINUOUS for static corpus
        primary_key="provision_id",
        embedding_source_column="provision_text",
        embedding_model_endpoint_name="databricks-gte-large-en",
    )
else:
    print(f"Index {INDEX_NAME} already exists")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    # Check if index is ready to sync
    status = index.describe().get("status", {})
    if status.get("ready", False):
        print("Index ready — triggering re-sync")
        index.sync()
    else:
        print(f"Index not ready yet (state: {status.get('message', 'unknown')})")

for attempt in range(360):
    try:
        detailed = index.describe().get("status", {})
        ready = detailed.get("ready", False)
        print(f"Index ready: {ready} | {detailed.get('message', '')}")
        if ready:
            break
    except Exception as e:
        # Handle transient 503s or network errors during polling
        if "503" in str(e) or "RetryError" in str(type(e).__name__):
            print(f"Transient error (attempt {attempt+1}/120): {e}")
        else:
            raise  # Re-raise unexpected errors
    time.sleep(30)
else:
    raise TimeoutError(f"Index {INDEX_NAME} did not become ready after 180 minutes")

Creating index workspace.default.ledgar_provisions_index...
Index ready: False | Delta sync Index creation is pending. Check latest status: https://dbc-6e816999-3b62.cloud.databricks.com/explore/data/workspace/default/ledgar_provisions_index
Index ready: False | Index is currently is in the process of syncing initial data. Check latest status: https://dbc-6e816999-3b62.cloud.databricks.com/explore/data/workspace/default/ledgar_provisions_index
Index ready: False | Index is currently is in the process of syncing initial data. Check latest status: https://dbc-6e816999-3b62.cloud.databricks.com/explore/data/workspace/default/ledgar_provisions_index
Index ready: False | Index is currently is in the process of syncing initial data. Check latest status: https://dbc-6e816999-3b62.cloud.databricks.com/explore/data/workspace/default/ledgar_provisions_index
Index ready: False | Index is currently is in the process of syncing initial data. Check latest status: https://dbc-6e816999-3b62.cloud.data

In [0]:
# Validate Retrieval

# Intake-style queries
test_queries = [
    "My former employer is refusing to pay severance promised in my contract",
    "The other company broke the confidentiality terms of our agreement",
    "Which state's law applies to a dispute over our software licensing deal?",
    "My landlord terminated the lease early without notice",
]

for q in test_queries:
    results = index.similarity_search(
        query_text=q,
        columns=["provision_id", "provision_text", "category"],
        num_results=3,
    )
    rows = results.get("result", {}).get("data_array", [])
    print(f"\nQUERY: {q}")
    for r in rows:
        print(f"  [{r[2]}] (score={r[-1]:.3f}) {r[1][:120]}...")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

QUERY: My former employer is refusing to pay severance promised in my contract
  [Enforcements] (score=0.641) Either party shall have the right specifically to enforce this Agreement and Release, except for provisions which subseq...
  [Benefits] (score=0.634) Each Eligible Employee who incurs a Severance shall be entitled, subject to Section 2.3, to receive the following paymen...
  [Non-Disparagement] (score=0.629) Executive agrees that he will not do or say, whether orally, in writing or in any other medium, anything inimical, derog...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

QUERY: The other company broke th

## Index Build Summary

- Corpus from **train split** of `default.ledgar_lexglue` → eval in 03 stays leakage-free
- Sampled mode: 80 provisions/category (~8k rows), all 100 labels represented; switch widget to full mode for the final run
- Delta Sync index uses managed embeddings (`databricks-gte-large-en`), `TRIGGERED` sync
- `02_agent`'s retrieval tool calls `similarity_search` on this index with the client's intake description